# Experiment 10: Supervised Review Sentiment Classification
## TF-IDF + Logistic Regression / LinearSVC / Naive Bayes with GridSearchCV
**Goal:** Build a robust sentiment classifier that predicts user review polarity (Positive vs. Negative) with calibrated confidence percentages.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_curve, auc


### 1. Load Cleaned Review Samples


In [ ]:
reviews_sample = pd.read_csv("../data/processed/reviews_clean.csv")
print("Cleaned Sample Shape:", reviews_sample.shape)
reviews_sample.head(3)


### 2. Train / Test Split (70/30 Stratified)


In [ ]:
X = reviews_sample['cleaned_review'].fillna('')
y = reviews_sample['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} samples")
print(f"Testing set:  {len(X_test)} samples")


### 3. Load Trained Model & Vectorizer


In [ ]:
vectorizer = joblib.load("../models/tfidf_vectorizer.pkl")
model = joblib.load("../models/sentiment_model.pkl")

X_test_tfidf = vectorizer.transform(X_test)
y_pred = model.predict(X_test_tfidf)
y_prob = model.predict_proba(X_test_tfidf)[:, 1]

print("Model Type:", type(model).__name__)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))


### 4. Classification Report & Confusion Matrix


In [ ]:
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(cm, cmap='Blues', alpha=0.8)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(x=j, y=i, s=f"{cm[i, j]:,}", va='center', ha='center', size='xx-large', weight='bold')

plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix on Test Split', fontsize=13, pad=15)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Negative', 'Positive'])
ax.set_yticklabels(['Negative', 'Positive'])
plt.colorbar(cax)
plt.show()


### 5. ROC Curve Analysis


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='#10b981', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='#6b7280', lw=1.5, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


### 6. Interactive Testing Example


In [ ]:
test_sentences = [
    "One of the greatest films ever made. Stunning visual poetry and hypnotic direction.",
    "Boring, painfully unoriginal, and filled with ridiculous dialogue from start to finish.",
    "Decent popcorn movie with good CGI, but the plot is thin."
]

for sentence in test_sentences:
    vec = vectorizer.transform([sentence])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    label = "POSITIVE" if pred == 1 else "NEGATIVE"
    conf = prob[pred] * 100
    print(f"Review: '{sentence}'")
    print(f"  -> Prediction: {label} ({conf:.1f}% confidence)\n")
